# DeepSeekMoE with Auxiliary-Loss-Free Load Balancing

- **DeepSeekMoE architecture** — finer-grained experts with $N_s$ shared experts and $N_r$ routed experts (Eq 1–6)
- **Auxiliary-loss-free load balancing** — per-expert bias terms for routing decisions (Eq 5)
- **Complementary sequence-wise auxiliary loss** — prevents extreme intra-sequence imbalance (Eq 7–10)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass


@dataclass
class MoEConfig:
    hidden_dim: int = 512          # model hidden dimension
    intermediate_dim: int = 1024   # FFN intermediate dimension
    num_shared_experts: int = 2    # N_s: number of shared experts
    num_routed_experts: int = 8    # N_r: number of routed experts
    top_k: int = 2                 # K_r: activated routed experts per token
    bias_update_speed: float = 0.001   # γ: bias update speed for load balancing
    balance_alpha: float = 0.0001  # α: coefficient for sequence-wise auxiliary loss

## Expert FFN (SwiGLU)

Each expert (shared or routed) is a standard Feed-Forward Network with SiLU-gated linear units:

$$\text{FFN}(\mathbf{x}) = W_{\text{down}} \left( \text{SiLU}(W_{\text{gate}}\,\mathbf{x}) \odot W_{\text{up}}\,\mathbf{x} \right) \tag{1}$$

In [8]:
class Expert(nn.Module):
    """A single FFN expert: two linear projections with SiLU gated activation (SwiGLU)."""

    def __init__(self, hidden_dim: int, intermediate_dim: int):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.up_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU: down(SiLU(gate(x)) * up(x))
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

## Top-K Router with Auxiliary-Loss-Free Load Balancing (Eq 2–5)

**Affinity score** — sigmoid dot-product between token hidden state and expert centroid:

$$s_{i,t} = \text{Sigmoid}(\mathbf{u}_t^\top \mathbf{e}_i) \tag{2}$$

**Top-K routing** — select the $K_r$ experts with the highest affinity:

$$g'_{i,t} = \begin{cases} s_{i,t}, & s_{i,t} \in \text{Topk}(\{s_{j,t} \mid 1 \leqslant j \leqslant N_r\},\, K_r) \\ 0, & \text{otherwise} \end{cases} \tag{3}$$

**Gating value** — normalize selected affinities:

$$g_{i,t} = \frac{g'_{i,t}}{\sum_{j=1}^{N_r} g'_{j,t}} \tag{4}$$

**Auxiliary-loss-free load balancing** — a bias term $b_i$ is added to affinity scores *only for routing*; the gating value still uses the original $s_{i,t}$:

$$g'_{i,t} = \begin{cases} s_{i,t}, & s_{i,t} + b_i \in \text{Topk}(\{s_{j,t} + b_j \mid 1 \leqslant j \leqslant N_r\},\, K_r) \\ 0, & \text{otherwise} \end{cases} \tag{5}$$

At each training step the bias is adjusted: $b_i \mathrel{-}= \gamma$ if expert $i$ is overloaded, $b_i \mathrel{+}= \gamma$ if underloaded.

In [ ]:
class TopKRouter(nn.Module):
    """
    Token-to-expert router with sigmoid affinity scoring and auxiliary-loss-free
    load balancing via per-expert bias terms.

    Affinity (Eq 2):  s_{i,t} = Sigmoid(u_t^T e_i)
    Routing  (Eq 3):  top-K selection over affinity scores
    Gating   (Eq 4):  normalize selected affinities across chosen experts
    Bias     (Eq 5):  bias b_i added to s_{i,t} for routing only (not gating value)
    """

    def __init__(self, hidden_dim: int, num_routed_experts: int, top_k: int,
                 bias_update_speed: float):
        super().__init__()
        self.top_k = top_k
        self.num_routed_experts = num_routed_experts
        self.bias_update_speed = bias_update_speed

        # Expert centroids e_i  (Eq 2)
        self.centroids = nn.Linear(hidden_dim, num_routed_experts, bias=False)

        # Per-expert bias terms b_i for load balancing (Eq 5) — not learnable via grad
        self.register_buffer("expert_bias", torch.zeros(num_routed_experts))

    def forward(self, u: torch.Tensor):
        """
        Args:
            u: (batch, seq_len, hidden_dim) — input hidden states

        Returns:
            gate_values:   (batch, seq_len, top_k) — normalized gating weights
            expert_indices: (batch, seq_len, top_k) — selected expert ids
            affinity_scores: (batch, seq_len, num_routed_experts) — raw sigmoid scores
        """
        # Eq 2: affinity scores via sigmoid
        affinity_scores = torch.sigmoid(self.centroids(u))  # (B, T, N_r)

        # Eq 5: add bias for routing decisions only
        biased_scores = affinity_scores + self.expert_bias  # (B, T, N_r)

        # Eq 3: top-K selection
        topk_vals, topk_indices = torch.topk(biased_scores, self.top_k, dim=-1)  # (B, T, K)

        # Gather the *original* affinity scores (without bias) for the selected experts
        # The gating value is derived from the original affinity s_{i,t}, not biased score
        selected_affinities = affinity_scores.gather(-1, topk_indices)  # (B, T, K)

        # Eq 4: normalize among selected experts to get gating values
        gate_values = selected_affinities / (selected_affinities.sum(dim=-1, keepdim=True) + 1e-9)

        return gate_values, topk_indices, affinity_scores

    @torch.no_grad()
    def update_bias(self, expert_counts: torch.Tensor, target_count: float):
        """
        Auxiliary-loss-free load balancing (Eq 5 discussion).
        After each training step, adjust bias: decrease for overloaded, increase for underloaded.

        Args:
            expert_counts: (num_routed_experts,) — total tokens routed to each expert this step
            target_count: expected balanced count per expert  (total_tokens * K_r / N_r)
        """
        overloaded = (expert_counts > target_count).float()
        underloaded = (expert_counts < target_count).float()
        self.expert_bias -= self.bias_update_speed * overloaded
        self.expert_bias += self.bias_update_speed * underloaded

## DeepSeekMoE Layer (Eq 6) & Sequence-Wise Auxiliary Loss (Eq 7–10)

**MoE FFN output** — residual + shared experts + gated routed experts:

$$\mathbf{h}'_t = \mathbf{u}_t + \sum_{i=1}^{N_s} \text{FFN}_i^{(s)}(\mathbf{u}_t) + \sum_{i=1}^{N_r} g_{i,t}\, \text{FFN}_i^{(r)}(\mathbf{u}_t) \tag{6}$$

---

**Complementary sequence-wise balance loss** — encourages balanced expert load within each sequence:

$$\mathcal{L}_{\text{Bal}} = \alpha \sum_{i=1}^{N_r} f_i\, P_i \tag{7}$$

where the **expert frequency** $f_i$ measures what fraction of tokens chose expert $i$:

$$f_i = \frac{N_r}{K_r T} \sum_{t=1}^{T} \mathbb{1}\!\left(s_{i,t} \in \text{Topk}(\{s_{j,t} \mid 1 \leqslant j \leqslant N_r\},\, K_r)\right) \tag{8}$$

the **normalized affinity** $s'_{i,t}$ distributes each token's scores across experts:

$$s'_{i,t} = \frac{s_{i,t}}{\sum_{j=1}^{N_r} s_{j,t}} \tag{9}$$

and the **expert score mass** $P_i$ averages the normalized affinity over the sequence:

$$P_i = \frac{1}{T} \sum_{t=1}^{T} s'_{i,t} \tag{10}$$

In [ ]:
class DeepSeekMoELayer(nn.Module):
    """
    DeepSeekMoE Feed-Forward Layer (Eq 6) with:
      - N_s shared experts (always active)
      - N_r routed experts (top-K_r selected per token)
      - Auxiliary-loss-free load balancing (Eq 5)
      - Complementary sequence-wise auxiliary loss (Eq 7-10)

    Output:  h'_t = u_t + Σ FFN_i^(s)(u_t) + Σ g_{i,t} FFN_i^(r)(u_t)
    """

    def __init__(self, config: MoEConfig):
        super().__init__()
        self.config = config
        self.num_routed_experts = config.num_routed_experts
        self.top_k = config.top_k
        self.balance_alpha = config.balance_alpha

        # Shared experts — always active for every token
        self.shared_experts = nn.ModuleList([
            Expert(config.hidden_dim, config.intermediate_dim)
            for _ in range(config.num_shared_experts)
        ])

        # Routed experts — selectively activated via top-K routing
        self.routed_experts = nn.ModuleList([
            Expert(config.hidden_dim, config.intermediate_dim)
            for _ in range(config.num_routed_experts)
        ])

        # Router
        self.router = TopKRouter(
            hidden_dim=config.hidden_dim,
            num_routed_experts=config.num_routed_experts,
            top_k=config.top_k,
            bias_update_speed=config.bias_update_speed,
        )

    def forward(self, u: torch.Tensor):
        """
        Args:
            u: (batch, seq_len, hidden_dim) — FFN input hidden states

        Returns:
            h_prime: (batch, seq_len, hidden_dim) — FFN output
            aux_loss: scalar — complementary sequence-wise balance loss
        """
        B, T, D = u.shape

        # ── Routing ──────────────────────────────────────────────
        gate_values, expert_indices, affinity_scores = self.router(u)
        # gate_values:     (B, T, K)
        # expert_indices:  (B, T, K)
        # affinity_scores: (B, T, N_r)

        # ── Shared experts (always active) ───────────────────────
        shared_output = sum(expert(u) for expert in self.shared_experts)  # (B, T, D)

        # ── Routed experts (sparse) ──────────────────────────────
        routed_output = torch.zeros_like(u)  # (B, T, D)

        # Flatten for efficient per-expert batching
        flat_u = u.view(-1, D)                                  # (B*T, D)
        flat_gate = gate_values.view(-1, self.top_k)            # (B*T, K)
        flat_indices = expert_indices.view(-1, self.top_k)      # (B*T, K)
        flat_output = torch.zeros_like(flat_u)                  # (B*T, D)

        for k in range(self.top_k):
            expert_ids = flat_indices[:, k]   # (B*T,)
            gate_k = flat_gate[:, k]          # (B*T,)

            for i in range(self.num_routed_experts):
                mask = (expert_ids == i)
                if mask.any():
                    tokens = flat_u[mask]                        # (n, D)
                    expert_out = self.routed_experts[i](tokens)  # (n, D)
                    flat_output[mask] += gate_k[mask].unsqueeze(-1) * expert_out

        routed_output = flat_output.view(B, T, D)

        # ── Eq 6: combine residual + shared + routed ───────────
        h_prime = u + shared_output + routed_output

        # ── Auxiliary loss (Eq 7-10) ─────────────────────────────
        aux_loss = self._sequence_wise_balance_loss(affinity_scores, expert_indices)

        return h_prime, aux_loss

    def _sequence_wise_balance_loss(self, affinity_scores: torch.Tensor,
                                     expert_indices: torch.Tensor) -> torch.Tensor:
        """
        Complementary sequence-wise auxiliary balance loss (Eq 7–10).

        Args:
            affinity_scores: (B, T, N_r) — raw sigmoid affinity scores
            expert_indices:  (B, T, K)   — indices of selected experts
        """
        B, T, N_r = affinity_scores.shape
        K_r = self.top_k

        # ── f_i (Eq 8): fraction of tokens routed to expert i ───
        # Build one-hot for selected experts and sum over top-k slot
        one_hot = F.one_hot(expert_indices, num_classes=N_r).float()  # (B, T, K, N_r)
        one_hot = one_hot.sum(dim=2)                                  # (B, T, N_r)
        # f_i = (N_r / (K_r * T)) * Σ_t 1(expert i selected for token t)
        f = (N_r / (K_r * T)) * one_hot.sum(dim=1)                   # (B, N_r)

        # ── s'_{i,t} (Eq 9): normalized affinity scores ─────────
        s_prime = affinity_scores / (affinity_scores.sum(dim=-1, keepdim=True) + 1e-9)  # (B, T, N_r)

        # ── P_i (Eq 10): mean normalized affinity per expert ─────
        P = s_prime.mean(dim=1)  # (B, N_r)

        # ── L_Bal (Eq 7): α * Σ_i f_i * P_i ────────────────────
        loss_per_batch = self.balance_alpha * (f * P).sum(dim=-1)  # (B,)
        return loss_per_batch.mean()

    @torch.no_grad()
    def update_load_balancing_bias(self, expert_indices: torch.Tensor, total_tokens: int):
        """
        Call after each training step to update the router's expert bias terms.

        Args:
            expert_indices: (B, T, K) — expert selections from the forward pass
            total_tokens: B * T
        """
        # Count how many tokens were routed to each expert
        flat = expert_indices.view(-1)
        expert_counts = torch.bincount(flat, minlength=self.num_routed_experts).float()
        target_count = total_tokens * self.top_k / self.num_routed_experts
        self.router.update_bias(expert_counts, target_count)

## Quick sanity check

In [5]:
torch.manual_seed(42)

config = MoEConfig(
    hidden_dim=256,
    intermediate_dim=512,
    num_shared_experts=2,
    num_routed_experts=8,
    top_k=2,
    bias_update_speed=0.001,
    balance_alpha=0.0001,
)

moe_layer = DeepSeekMoELayer(config)
print(f"Total parameters: {sum(p.numel() for p in moe_layer.parameters()):,}")

# Dummy input: batch=4, seq_len=16, hidden_dim=256
u = torch.randn(4, 16, config.hidden_dim)

# Forward pass
h_prime, aux_loss = moe_layer(u)
print(f"Input shape:  {u.shape}")
print(f"Output shape: {h_prime.shape}")
print(f"Aux loss:     {aux_loss.item():.6f}")

# Simulate one load-balancing bias update
_, expert_indices, _ = moe_layer.router(u)
moe_layer.update_load_balancing_bias(expert_indices, total_tokens=4 * 16)
print(f"Expert bias after update: {moe_layer.router.expert_bias}")
print(f"Bias range: [{moe_layer.router.expert_bias.min():.4f}, {moe_layer.router.expert_bias.max():.4f}]")

Total parameters: 3,934,208
Input shape:  torch.Size([4, 16, 256])
Output shape: torch.Size([4, 16, 256])
Aux loss:     0.000102
Expert bias after update: tensor([ 0.0010, -0.0010, -0.0010,  0.0010,  0.0010,  0.0010, -0.0010, -0.0010])
Bias range: [-0.0010, 0.0010]
